# 1. Make classification data and get it ready

In [21]:
from sklearn.datasets import make_circles

#Make 1000 samples
n_samples = 1000

#Create circles
X,y = make_circles(n_samples, noise = 0.03, random_state = 42)

len(X), len(y)

(1000, 1000)

In [22]:
print(f"First 5 samples of X\n {X[:5]}")
print(f"First 5 samples of y \n{y[:5]}")

First 5 samples of X
 [[ 0.75424625  0.23148074]
 [-0.75615888  0.15325888]
 [-0.81539193  0.17328203]
 [-0.39373073  0.69288277]
 [ 0.44220765 -0.89672343]]
First 5 samples of y 
[1 1 1 1 0]


In [23]:
#Make DataFrame of circle data
import pandas as pd
circles = pd.DataFrame({"X1":X[:,0],
                        "X2":X[:,1],
                        "label":y})
circles.head(10)

,X1,X2,label
0,0.754246,0.231481,1
1,-0.756159,0.153259,1
2,-0.815392,0.173282,1
3,-0.393731,0.692883,1
4,0.442208,-0.896723,0
5,-0.479646,0.676435,1
6,-0.013648,0.803349,1
7,0.771513,0.147760,1
8,-0.169322,-0.793456,1
9,-0.121486,1.021509,0


In [24]:
#Visualize
import matplotlib.pyplot as plt
plt.scatter(x=X[:,0],
            y=X[:,1],
            c=y,
            cmap = plt.cm.RdYlBu)

ModuleNotFoundError: No module named 'matplotlib'

### 1.1 Check input and Output shapes

In [5]:
X.shape,y.shape

((1000, 2), (1000,))

In [6]:
#View first example of features and labels
X_sample = X[0]
y_sample = y[0]
print(f"X sample: {X_sample}")
print(f"y sample: {y_sample}")
print(f"X sample shape: {X_sample.shape}")
print(f"y sample shape: {y_sample.shape}")


X sample: [0.75424625 0.23148074]
y sample: 1
X sample shape: (2,)
y sample shape: ()


### 1.2 Turn data into tensors and create train and test splits

In [7]:
#Turn data into tensors
import torch

X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)



In [8]:
X[:5], y[:5]
type(X), X.dtype,y.dtype

(torch.Tensor, torch.float32, torch.float32)

In [9]:
#Split data into training and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 


In [10]:
len(X_train), len(y_train), len(X_test), len(y_test)

(800, 800, 200, 200)

In [25]:
#2. Building a model

from torch import nn
#Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device
import torch
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")


0
No CUDA device


1. Subclasses nn.Module (almost all PyTorch models are subclasses of nn.Module).
2. Creates 2 nn.Linear layers in the constructor capable of handling the input and output shapes of X and y.
3. Defines a forward() method containing the forward pass computation of the model.
4. Instantiates the model class and sends it to the target device.

In [12]:
#1. Construct a model that subclasses nn.Module

class CircleModelV0(nn.Module):
    def __init__(self):
        super().__init__()
        # 2. Create 2 nn.Linear layers capable of handling the shapes of our data
        self.layer_1 = nn.Linear(in_features = 2,out_features=5) #Take in 2 features and upscales it to 5 features
        self.layer_2 = nn.Linear(in_features = 5, out_features = 1) #takes in 5 features from prev layers and ops a single feature(same shape as y)

        # self.two_linear_layers = nn.Sequential(
        #     nn.Linear(in_features = 2,out_features = 5),
        #     nn.Linear(in_features = 5, out_features = 1)
        # )

    #3. Define a forward() method that outlines the forward pass
    def forward(self, x):
        self.layer_2(self.layer_1(x)) #x-> layer_1 ->layer_2->output
        # return two_linear_layers(x) 
    
#4. Instantiate an instance of the model class and send it to the target device
model_0 = CircleModelV0().to(device)
model_0,device

(CircleModelV0(
   (layer_1): Linear(in_features=2, out_features=5, bias=True)
   (layer_2): Linear(in_features=5, out_features=1, bias=True)
 ),
 'cpu')

In [13]:
next(model_0.parameters()).device

device(type='cpu')

In [14]:
#replicate the model above using nn.Sequential()
model_0 = nn.Sequential(
    nn.Linear(in_features=2, out_features=5),
    nn.Linear(in_features=5, out_features=1)
).to(device)    
model_0

Sequential(
  (0): Linear(in_features=2, out_features=5, bias=True)
  (1): Linear(in_features=5, out_features=1, bias=True)
)

In [15]:
model_0.state_dict() #returns a dictionary of all the parameters in the model


OrderedDict([('0.weight',
              tensor([[-0.4320, -0.3031],
                      [-0.5859, -0.4371],
                      [-0.1098, -0.2562],
                      [-0.2306, -0.0943],
                      [-0.5956, -0.0983]])),
             ('0.bias', tensor([-0.3507,  0.0462,  0.5381,  0.1897, -0.5863])),
             ('1.weight',
              tensor([[-0.2558,  0.2614,  0.1730,  0.1630,  0.3247]])),
             ('1.bias', tensor([0.1415]))])

In [16]:
#Make predictions
with torch.inference_mode():
    untrained_preds = model_0(X_test.to(device))
print(f"Length of untrained predictions: {len(untrained_preds)}, Shape: {untrained_preds.shape}")   
print(f"Length of test samples: {len(X_test)}, Shape: {X_test.shape}")
print(f"First 10 untrained predictions: {untrained_preds[:10]}")
print(f"First 10 labels: {y_test[:10]}")

Length of untrained predictions: 200, Shape: torch.Size([200, 1])
Length of test samples: 200, Shape: torch.Size([200, 2])
First 10 untrained predictions: tensor([[ 0.1991],
        [ 0.0492],
        [ 0.4229],
        [ 0.1363],
        [ 0.1545],
        [ 0.0651],
        [-0.1442],
        [-0.1412],
        [ 0.4339],
        [ 0.0361]])
First 10 labels: tensor([1., 0., 1., 0., 1., 1., 0., 0., 1., 0.])


In [17]:
X_test[:10], y_test[:10]

(tensor([[-0.3752,  0.6827],
         [ 0.0154,  0.9600],
         [-0.7028, -0.3147],
         [-0.2853,  0.9664],
         [ 0.4024, -0.7438],
         [ 0.6323, -0.5711],
         [ 0.8561,  0.5499],
         [ 1.0034,  0.1903],
         [-0.7489, -0.2951],
         [ 0.0538,  0.9739]]),
 tensor([1., 0., 1., 0., 1., 1., 0., 0., 1., 0.]))

### 2.1 Setup loss function and optimizer

In [18]:
#Setup a loss function
import torch.nn as nn
loss_fun = nn.BCEWithLogitsLoss() #Binary Cross Entropy with logits loss function

optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.1) #Stochastic Gradient Descent optimizer

In [19]:
#Calculate accuracy - out of 100 examples, what percentages does our model get right

def accuracy_fun(y_true,y_pred):
    correct = torch.eq(y_true, y_pred).sum().item() #counts the number of correct predictions
    acc = correct / len(y_pred) * 100 #calculates the accuracy as a percentage
    return acc

### 3. train model

In [20]:
import sys
print(sys.version)

3.10.18 (main, Jun  4 2025, 17:33:50) [MSC v.1943 64 bit (AMD64)]
